# A8 — semi-implicit Euler + Verlet

Semi-implicit stabilises. Verlet is nicer for constraints.

In [1]:
import numpy as np

def simulate_semi_implicit(self, dt, g, damp=0.01):
    for s in self.springs:
        d = s.b.pos - s.a.pos
        L = np.linalg.norm(d)
        f = s.k * (d / L) * (L - s.rest)
        s.a.force += f
        s.b.force -= f
    for m in self.masses:
        if m.pinned:
            m.force[:] = 0
            continue
        m.force += g
        m.vel = (1 - damp) * (m.vel + m.force * dt)
        m.pos = m.pos + m.vel * dt  # uses updated velocity -> semi-implicit
        m.force[:] = 0

def simulate_verlet(self, dt, g, damp=0.00005):
    for s in self.springs:
        d = s.b.pos - s.a.pos
        L = np.linalg.norm(d)
        f = s.k * (d / L) * (L - s.rest)
        s.a.force += f
        s.b.force -= f
    for m in self.masses:
        if m.pinned:
            m.force[:] = 0
            continue
        accel = m.force + g
        new = m.pos + (1 - damp) * (m.pos - m.last) + accel * dt * dt
        m.last = m.pos.copy()
        m.pos = new
        m.force[:] = 0
